In [32]:
import requests as req
from bs4 import BeautifulSoup as B
url = 'https://www.theeducatedbarfly.com/winter-daiquiri/'
header = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'}
resp = req.get(url, headers=header)
if resp.status_code == 200:
    soup = B(resp.text, 'html.parser')
    for cocktail_name in soup.find_all('div', class_='tdb-block-inner td-fix-index'):
        name = cocktail_name.find('h1')
        if name:
            print(name.text.strip())


Winter Daiquiri


In [31]:


if resp.status_code == 200:
    soup = B(resp.text, 'html.parser')
    
    # 查找所有具有指定 class 的 <span> 元素
    ingredients = soup.find_all('span', class_='wprm-recipe-ingredient-name')
    
    for ingredient in ingredients:
        # 提取 <a> 標籤的文字內容
        ingredient_name = ingredient.find('a')
        if ingredient_name:  # 確保 <a> 存在
            print(ingredient_name.text.strip())

Simple Syrup
Lime Juice
Blackstrap Rum


In [44]:
import pandas as pd


In [45]:
cocktail_name_df

,Name
0,Harry’s Midnight Snack
1,Bushwick
2,Winter Daiquiri
3,Mundo Perdido
4,Formidable Dragon
...,...
772,Bee’s Knees
773,Negroni
774,Daisy
775,Manhattan


In [6]:
import requests as req
from bs4 import BeautifulSoup as B
import pandas as pd
import time
import random
builder_url = 'https://www.theeducatedbarfly.com/cocktail-builder/'
header = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'}
resp = req.get(builder_url, headers=header)

cocktail_name_list = []

if resp.status_code == 200:
    soup = B(resp.text, 'html.parser')
    for cocktail_name in soup.find_all('div', class_="wpupg-item-title wpupg-block-text-bold"):
        name = cocktail_name.text.strip()
        if name:
            cocktail_name_list.append(name)
    time.sleep(random.uniform(1,3))

cocktail_name_df = pd.DataFrame(cocktail_name_list, columns=['Name'])
cocktail_data = []

for name in cocktail_name_df['Name']:
    urls = f'https://www.theeducatedbarfly.com/{name.replace(" ", "-").lower()}/'
    resp = req.get(urls, headers=header)

    if resp.status_code == 200:
        soup = B(resp.text, 'html.parser')
    
        # 查找所有具有指定 class 的 <span> 元素
        ingredients = soup.find_all('span', class_='wprm-recipe-ingredient-name')
        ingredient_name_list = []

        for ingredient in ingredients:
        # 提取 <a> 標籤的文字內容
            ingredient_name = ingredient.find('a')
            if ingredient_name:  # 確保 <a> 存在
                ingredient_name_list.append(ingredient_name.text.strip())

        cocktail_data.append({'Cocktail Name': name, 'Ingredients': ", ".join(ingredient_name_list)})
    time.sleep(random.uniform(1,3))

cocktail_df = pd.DataFrame(cocktail_data)



In [8]:
cocktail_df['Ingredients'] = cocktail_df['Ingredients'].apply(
    lambda x: ", ".join([ingredient.lower() for ingredient in x.split(", ")])
)

cocktail_df.to_csv('cocktail_ingredients.csv', index=False)

cocktail_df.head()

,Cocktail Name,Ingredients
0,Bushwick,"amaro lucano, maraschino liqueur, sweet vermou..."
1,Winter Daiquiri,"simple syrup, lime juice, blackstrap rum"
2,Mundo Perdido,"lemon juice, demerara syrup, cinnamon syrup, a..."
3,Formidable Dragon,"lemon juice, lime juice, molasses syrup, honey..."
4,Tradewinds,"lemon juice, coconut cream, apricot liqueur, a..."


In [17]:
# 提取所有唯一的成分
all_ingredients = set()
for ingredients in cocktail_df['Ingredients']:
    # 將成分分割成清單，並加入集合中
    all_ingredients.update(ingredient.strip().lower() for ingredient in ingredients.split(','))

# 將成分集合轉為排序列表（列名順序一致）
all_ingredients = sorted(all_ingredients)

# 初始化矩陣 DataFrame
matrix = pd.DataFrame(0, index = cocktail_df['Cocktail Name'], columns = all_ingredients)

# 填充矩陣
for idx, row in cocktail_df.iterrows():
    ingredients = [ingredient.strip().lower() for ingredient in row['Ingredients'].split(',')]
    for ingredient in ingredients:
        if ingredient in matrix.columns:
            matrix.loc[row['Cocktail Name'], ingredient] = 1

# 查看結果矩陣

matrix.to_csv('cocktail_builder_matrix.csv', index=True, encoding='utf-8')


In [18]:
matrix

,,12 rum blend infinity bottle,151 rum,absinthe,absinthe bitters,absinthe rinse,absinthe spritz,acacia honey syrup,acid adjusted pineapple juice,acid phosphate,...,wine,woodford reserve,xanthan gum,xocolatl mole bitters,yellow chartreuse,yellow pepper juice,yuzu juice,yuzu puree,yuzu soda,zucca rabarbaro
Cocktail Name,,,,,,,,,,,,,,,,,,,,,
Bushwick,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Winter Daiquiri,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Mundo Perdido,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Formidable Dragon,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Tradewinds,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Jack Rose,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Negroni,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Daisy,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
data_matrix = pd.read_csv('cocktail_builder_matrix.csv')

In [45]:
data_matrix = data_matrix.drop(columns=['Unnamed: 1'])

In [46]:
data_matrix

,Cocktail Name,12 rum blend infinity bottle,151 rum,absinthe,absinthe bitters,absinthe rinse,absinthe spritz,acacia honey syrup,acid adjusted pineapple juice,acid phosphate,...,wine,woodford reserve,xanthan gum,xocolatl mole bitters,yellow chartreuse,yellow pepper juice,yuzu juice,yuzu puree,yuzu soda,zucca rabarbaro
0,Bushwick,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Winter Daiquiri,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Mundo Perdido,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Formidable Dragon,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Tradewinds,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
674,Jack Rose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
675,Negroni,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
676,Daisy,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
677,Manhattan,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
import pandas as pd
import numpy as np
from sklearn.metrics import jaccard_score

# 設定數據矩陣（剔除雞尾酒名稱列，僅保留二進制成分矩陣）
binary_matrix = data_matrix.set_index('Cocktail Name')

# 初始化空的相似度矩陣
n = binary_matrix.shape[0]
similarity_matrix = np.zeros((n, n))

# 計算 Jaccard 相似度
for i in range(n):
    for j in range(n):
        similarity_matrix[i, j] = jaccard_score(
            binary_matrix.iloc[i], binary_matrix.iloc[j]
        )

# 將相似度矩陣轉為 DataFrame
jaccard_similarity_df = pd.DataFrame(
    similarity_matrix,
    index=binary_matrix.index,
    columns=binary_matrix.index
)

c:\Users\A408\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 due to no true or predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [48]:
jaccard_similarity_df

Cocktail Name,Bushwick,Winter Daiquiri,Mundo Perdido,Formidable Dragon,Tradewinds,Tipsy Jitters,Cafe Mexicana,Sleepwalker,Damn Fine Coffee Martini,Daily Rituals,...,Aperol Spritz,Monte Carlo,French 75,Champagne Cocktail,Boulevardier,Jack Rose,Negroni,Daisy,Manhattan,Sazerac
Cocktail Name,,,,,,,,,,,,,,,,,,,,,
Bushwick,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.142857,0.000000,0.000000,0.142857,0.000000,0.142857,0.000000,0.333333,0.111111
Winter Daiquiri,0.000000,1.000000,0.000000,0.076923,0.000000,0.111111,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.200000,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000
Mundo Perdido,0.000000,0.000000,1.000000,0.133333,0.083333,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.125000,0.000000,0.000000,0.111111,0.000000,0.100000,0.000000,0.000000
Formidable Dragon,0.000000,0.076923,0.133333,1.000000,0.125000,0.000000,0.0,0.0,0.000000,0.000000,...,0.071429,0.000000,0.076923,0.000000,0.000000,0.071429,0.000000,0.142857,0.000000,0.062500
Tradewinds,0.000000,0.000000,0.083333,0.125000,1.000000,0.000000,0.0,0.0,0.000000,0.076923,...,0.000000,0.000000,0.111111,0.000000,0.000000,0.000000,0.000000,0.090909,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Jack Rose,0.000000,0.166667,0.111111,0.071429,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
Negroni,0.142857,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.166667,0.000000,0.333333,0.000000,1.000000,0.125000,0.142857,0.000000
Daisy,0.000000,0.000000,0.100000,0.142857,0.090909,0.000000,0.0,0.0,0.111111,0.000000,...,0.125000,0.125000,0.333333,0.000000,0.000000,0.000000,0.125000,1.000000,0.000000,0.100000
